# Bi-objective Traveling Salesman Problem (TSP)
## Genetic Algorithm Solution for 15 Cities

Minimizes:
- f1: Total distance
- f2: Total cost
- F = f1 + f2

## 1. Imports and Parameters

In [ ]:
from __future__ import annotations
import csv
import os
import re
import random
from typing import List, Union

In [ ]:
class Parameters:
    """Configuration parameters for the genetic algorithm."""
    individuals_nb: int = 300              # Number of individuals per generation
    generations_max_nb: int = 500          # Maximum number of generations
    initial_genes_nb: int = 15             # Number of cities
    min_fitness: int = 0                   # Target fitness value
    mutations_rate: float = 0.15           # Mutation rate
    mutations_add_rate: float = 0.20       # Gene addition rate
    mutation_delete_rate: float = 0.10    # Gene deletion rate
    crossover_rate: float = 0.70           # Crossover rate

## 2. TSP Problem Definition (Load from CSV)

In [ ]:
class TSP:
    """
    Bi-objective Traveling Salesman Problem for 15 cities.
    Data is loaded from CSV files.
    """
    cities: List[str] = []
    distances: List[List[int]] = []
    costs: List[List[int]] = []

    @classmethod
    def load_data(cls, distances_file: str = "distances.csv", costs_file: str = "cities.csv") -> None:
        """Load cities, distances and costs from CSV files."""
        # Load distances matrix
        with open(distances_file, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            header = next(reader)
            cls.cities = header[1:]
            cls.distances = []
            for row in reader:
                if row:
                    cls.distances.append([int(val) for val in row[1:]])

        # Load costs matrix
        with open(costs_file, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)
            cls.costs = []
            for row in reader:
                if row:
                    cls.costs.append([int(val) for val in row[1:]])

    @classmethod
    def distance(cls, city1: str, city2: str) -> int:
        """Get the distance between two cities."""
        return cls.distances[cls.cities.index(city1)][cls.cities.index(city2)]

    @classmethod
    def cost(cls, city1: str, city2: str) -> int:
        """Get the cost between two cities."""
        return cls.costs[cls.cities.index(city1)][cls.cities.index(city2)]


# Load data from CSV files
TSP.load_data()
print(f"Loaded {len(TSP.cities)} cities: {', '.join(TSP.cities)}")

## 3. Gene Class

In [ ]:
class Gene:
    """Represents a city in the tour."""
    city: str

    def __init__(self, city: Union[str, "Gene"]) -> None:
        self.city = str(city)

    def __str__(self):
        return self.city

    def __eq__(self, other: "Gene") -> bool:
        return str(self) == str(other)

    def __ne__(self, other: "Gene") -> bool:
        return str(self) != str(other)

    def distance(self, gene: "Gene") -> int:
        return TSP.distance(self.city, gene.city)

    def cost(self, gene: "Gene") -> int:
        return TSP.cost(self.city, gene.city)

## 4. Individual Class (Solution Representation)

In [ ]:
class Individual:
    """Represents a candidate solution (tour)."""
    fitness: int = -1      # F = f1 + f2
    distance: int = -1     # f1: total distance
    cost: int = -1         # f2: total cost
    genome: List[Gene]

    def __init__(self, genome: List[Gene], mutate: bool = False) -> None:
        self.genome = genome
        if mutate:
            self.mutate()

    @classmethod
    def construct_from_cities(cls) -> "Individual":
        """Create a new individual with a random ordering of cities."""
        cities = TSP.cities[:]
        random.shuffle(cities)
        return cls(genome=[Gene(city) for city in cities])

    @classmethod
    def construct_from_father(cls, father: "Individual") -> "Individual":
        """Create a new individual by copying the father's genome with possible mutation."""
        return cls(genome=[Gene(gene) for gene in father.genome], mutate=True)

    @classmethod
    def construct_from_parents(cls, father: "Individual", mother: "Individual") -> "Individual":
        """Create a new individual using two-point crossover."""
        size = len(father.genome)

        # Select two random crossover points
        point1 = random.randint(1, size - 2)
        point2 = random.randint(point1 + 1, size - 1)

        # Start with father's genome
        child_genome = [Gene(gene) for gene in father.genome]

        # Copy segment from mother between point1 and point2
        for i in range(point1, point2):
            child_genome[i] = Gene(mother.genome[i])

        # Find duplicates and missing cities
        cities_in_child = [str(g) for g in child_genome]
        all_cities = set(TSP.cities)

        seen = set()
        duplicates = []
        for i, city in enumerate(cities_in_child):
            if city in seen:
                duplicates.append(i)
            else:
                seen.add(city)

        missing = list(all_cities - seen)
        random.shuffle(missing)

        # Replace duplicates with missing cities
        for i, dup_idx in enumerate(duplicates):
            child_genome[dup_idx] = Gene(missing[i])

        return cls(genome=child_genome, mutate=True)

    def mutate(self) -> None:
        """Apply mutation by swapping two random genes."""
        if random.random() < Parameters.mutations_rate:
            if len(self.genome) >= 2:
                i, j = random.sample(range(len(self.genome)), 2)
                self.genome[i], self.genome[j] = self.genome[j], self.genome[i]

    def evaluate(self) -> None:
        """Calculate the bi-objective fitness: F = f1 + f2."""
        self.distance = 0
        self.cost = 0

        for i in range(len(self.genome) - 1):
            self.distance += self.genome[i].distance(self.genome[i + 1])
            self.cost += self.genome[i].cost(self.genome[i + 1])

        # Add return to starting city
        self.distance += self.genome[-1].distance(self.genome[0])
        self.cost += self.genome[-1].cost(self.genome[0])

        self.fitness = self.distance + self.cost

    def get_tour_string(self) -> str:
        """Get the tour as a comma-separated string."""
        cities = [str(gene) for gene in self.genome]
        cities.append(cities[0])
        return ",".join(cities)

    def __str__(self) -> str:
        route = " -> ".join(str(gene) for gene in self.genome)
        start = str(self.genome[0])
        return f"F={self.fitness} (Distance={self.distance}, Cost={self.cost}) | {route} -> {start}"

## 5. Evolutionary Process (Genetic Algorithm)

In [ ]:
class EvolutionaryProcess:
    """Genetic algorithm for solving TSP."""
    population: List[Individual]
    generation: int
    best_individual: Individual
    best_generation: int

    def __init__(self) -> None:
        self.generation = 0
        self.best_generation = 0
        self.best_individual = None
        self.population = [
            Individual.construct_from_cities()
            for _ in range(Parameters.individuals_nb)
        ]

    def run(self) -> Individual:
        """Run the evolutionary process."""
        print(f"Starting GA with {Parameters.individuals_nb} individuals")
        print(f"Maximum generations: {Parameters.generations_max_nb}")
        print("-" * 60)

        while self.generation < Parameters.generations_max_nb:
            for individual in self.population:
                individual.evaluate()

            self.population.sort(key=lambda ind: ind.fitness)

            current_best = self.population[0]
            if self.best_individual is None or current_best.fitness < self.best_individual.fitness:
                self.best_individual = current_best
                self.best_generation = self.generation
                print(f"Gen {self.generation:4d}: New best F={current_best.fitness} (dist={current_best.distance}, cost={current_best.cost})")

            if Parameters.min_fitness > 0 and current_best.fitness <= Parameters.min_fitness:
                print(f"\nTarget fitness reached at generation {self.generation}!")
                break

            self.population = self.next_generation()
            self.generation += 1

        print("-" * 60)
        print(f"\nBest solution found at generation {self.best_generation}:")
        print(self.best_individual)

        return self.best_individual

    def next_generation(self) -> List[Individual]:
        """Create the next generation."""
        new_population = [self.population[0]]  # Elitism

        while len(new_population) < Parameters.individuals_nb:
            if random.random() < Parameters.crossover_rate:
                father = self.select()
                mother = self.select()
                offspring = Individual.construct_from_parents(father, mother)
            else:
                parent = self.select()
                offspring = Individual.construct_from_father(parent)

            new_population.append(offspring)

        return new_population

    def select(self) -> Individual:
        """Tournament selection."""
        tournament_size = min(5, len(self.population))
        tournament = random.sample(self.population, tournament_size)
        tournament.sort(key=lambda ind: ind.fitness)
        return tournament[0]

## 6. Solution Management (XML Save/Load)

In [ ]:
def load_solutions(filename: str = "solution.xml") -> tuple:
    """Load existing solutions from XML file."""
    if not os.path.exists(filename):
        return -1, []

    try:
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read()

        fitness_match = re.search(r'<BestFitness>(\d+)</BestFitness>', content)
        if not fitness_match:
            fitness_match = re.search(r'<Fitness>(\d+)</Fitness>', content)

        if not fitness_match:
            return -1, []

        best_fitness = int(fitness_match.group(1))
        tours = re.findall(r'<Tour>([^<]+)</Tour>', content)

        return best_fitness, tours
    except Exception:
        return -1, []


def save_solutions_xml(fitness: int, distance: int, cost: int, tours: list, filename: str = "solution.xml") -> None:
    """Save all best solutions to an XML file."""
    solutions_xml = ""
    for i, tour in enumerate(tours, 1):
        solutions_xml += f'''
    <Solution id="{i}">
        <Tour>{tour}</Tour>
        <Distance>{distance}</Distance>
        <Cost>{cost}</Cost>
    </Solution>
'''

    xml_content = f'''<?xml version="1.0" encoding="UTF-8"?>
<TSP_Solutions>
    <BestFitness>{fitness}</BestFitness>
    <TotalSolutions>{len(tours)}</TotalSolutions>
{solutions_xml}
</TSP_Solutions>
'''
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(xml_content)
    print(f"Solution saved to {filename}")

## 7. Run the Genetic Algorithm

In [ ]:
print("=" * 70)
print("BI-OBJECTIVE TRAVELING SALESMAN PROBLEM")
print("Minimize F = f1 (distance) + f2 (cost)")
print("=" * 70)
print()
print(f"Problem: {len(TSP.cities)} cities")
print(f"Cities: {', '.join(TSP.cities)}")
print()

# Load previous best
previous_best, existing_tours = load_solutions("solution.xml")
if previous_best > 0:
    print(f"Previous best fitness: {previous_best}")
    print(f"Existing solutions: {len(existing_tours)}")
    print()

# Run GA
process = EvolutionaryProcess()
best = process.run()

## 8. Save Results

In [ ]:
new_tour = best.get_tour_string()

print()
print("=" * 70)

if previous_best < 0:
    save_solutions_xml(best.fitness, best.distance, best.cost, [new_tour])
    print("FIRST SOLUTION SAVED")
    print(f"Tour: {new_tour}")
    print(f"F={best.fitness} (Distance={best.distance}, Cost={best.cost})")

elif best.fitness < previous_best:
    save_solutions_xml(best.fitness, best.distance, best.cost, [new_tour])
    print(f"NEW BEST FOUND! (improved from {previous_best} to {best.fitness})")
    print(f"Tour: {new_tour}")
    print(f"F={best.fitness} (Distance={best.distance}, Cost={best.cost})")

elif best.fitness == previous_best:
    if new_tour not in existing_tours:
        existing_tours.append(new_tour)
        save_solutions_xml(best.fitness, best.distance, best.cost, existing_tours)
        print(f"NEW TOUR FOUND WITH SAME FITNESS ({best.fitness})!")
        print(f"New tour: {new_tour}")
        print(f"Total solutions: {len(existing_tours)}")
    else:
        print(f"TOUR ALREADY EXISTS (F={best.fitness})")
        print(f"Total solutions: {len(existing_tours)}")

else:
    print(f"NO IMPROVEMENT (current: {best.fitness}, best: {previous_best})")
    print(f"Keeping {len(existing_tours)} solution(s) with F={previous_best}")

print("=" * 70)

## 9. View Current Best Solutions

In [ ]:
# Display all saved solutions
best_fitness, tours = load_solutions("solution.xml")
print(f"Best Fitness: {best_fitness}")
print(f"Total Solutions: {len(tours)}")
print()
for i, tour in enumerate(tours, 1):
    print(f"Solution {i}: {tour}")